# ChakraModel R5: encoder / decoder ablation

**The question.** Transformer models flag 83–85% of polyp-free frames with high confidence; the ResNet U-Net flags 22%. Is that caused by the transformer **encoder**, or by a **decoder** that predicts at 1/4 resolution and upsamples, instead of decoding progressively to full resolution?

**The design.** 3 encoders (resnet34, mit_b2, pvt_v2_b2) x 3 decoders (U-Net, FPN, SegFormer MLP head), one training recipe, 3 seeds. Three cells already exist from E1 (unet_r34, seg_mit_b2 = "segformer_b2", fpn_pvtv2b2), so this notebook trains the 6 missing cells: 18 runs. Each run is scored on the 5 polyp tests plus 4,275 PolypGen and 2,947 HyperKvasir polyp-free frames, so Dice and false alarms come from the same run.

**How to read the result.** If false alarms track the decoder (FPN/MLP high, U-Net low) whatever the encoder, the cause is the decoder. If they track the encoder, the cause is the encoder. Either answer is a finding; the first one is the more useful one, because it is a design rule.

**Before you run, set:**
- **Settings:** Accelerator = **GPU T4 x2**, Internet = **On**.
- **Inputs (Add Input), three datasets:** chakra-leakbench-v1, chakra-leakbench-e4-v1, chakra-leakbench-r2-v1-22-09-14-50
- **Run:** Save Version -> Save & Run All (Commit).

**Split over two accounts if quota is tight:** `HALF = "A"` (jobs 100-108) in one, `HALF = "B"` (109-117) in the other; about 3 h each. `HALF = ""` runs all 18 (about 6 h).

**Expected log:** the three input paths, `r5_ablation.csv {...}` with 1,228 train / 222 val, `jobs 18 100 - 117`, then six `builds:` lines with parameter counts (21.9 M - 28.1 M), then `=== job 100 ...`. Ends with `[0, 0]` and `finished runs 18`. Re-committing skips finished runs.

**Tested before shipping:** all six architectures build and produce 352x352 output, and the manifest/job construction was dry-run against a mock of the Kaggle input layout.

In [1]:
import subprocess
subprocess.run('pip -q install segmentation-models-pytorch==0.5.0 timm', shell=True)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 8.1 MB/s eta 0:00:00


CompletedProcess(args='pip -q install segmentation-models-pytorch==0.5.0 timm', returncode=0)

In [2]:
# R5 decoder/encoder ablation: 3 encoders x 3 decoders, one recipe, 3 seeds.
# Question: are the confident false alarms on polyp-free frames a property of the transformer ENCODER,
# or of a decoder that predicts at low resolution without a full-resolution skip?
#   encoders: resnet34 (CNN) | mit_b2 (SegFormer hierarchical ViT) | pvt_v2_b2 (PVTv2)
#   decoders: unet (progressive upsampling with skips, ends at full resolution)
#             fpn  (predicts at 1/4 resolution, then a single x4 upsample)
#             seg  (SegFormer MLP head: fuse at 1/4, then x4 upsample)
# Three corners already exist from E1 (unet_r34, segformer_b2 = seg_mit_b2, fpn_pvtv2b2), so this run
# trains the 6 missing cells. Each run is evaluated on the 5 polyp tests + 4,275 PolypGen negatives
# + 2,947 HyperKvasir negatives, so Dice and false alarms come out of the same run.
# Inputs: chakra-leakbench-v1, chakra-leakbench-e4-v1, chakra-leakbench-r2-v1-22-09-14-50.
# HALF = "" runs all 18 (~6 h). HALF = "A" runs the first 9, "B" the last 9 (~3 h each, two accounts).
HALF = ""
import glob, os, csv, subprocess
IN = "/kaggle/input"
def find_dir(marker, depth=6):
    for d in range(1, depth + 1):
        h = glob.glob(os.path.join(IN, *(["*"] * d), marker))
        if h: return os.path.dirname(h[0])
    raise FileNotFoundError(marker)
S1 = find_dir("fixed_polyp_dataset_v1"); E4 = find_dir("negatives_hyperkvasir_v1"); R2 = find_dir("runs_e1")
ROOT = "/kaggle/working/root"; W = "/kaggle/working/leakbench"; os.makedirs(ROOT, exist_ok=True)
print("S1", S1, "\nE4", E4, "\nCODE", R2, flush=True)
for src in (S1, E4):
    for e in os.listdir(src):
        if e not in ("leakbench", "runs_e1") and not os.path.exists(f"{ROOT}/{e}"): os.symlink(f"{src}/{e}", f"{ROOT}/{e}")
subprocess.run(f"rm -rf {W} && cp -r {R2}/leakbench {W}", shell=True, check=True)

# --- add the 9 encoder/decoder combinations to build_model (text patch, verified below)
NEW = '''    ENC = {"r34": "resnet34", "mit_b2": "mit_b2", "pvtv2b2": "tu-pvt_v2_b2"}
    if "_" in name and name.split("_", 1)[0] in ("unet", "fpn", "seg"):
        dec, enc = name.split("_", 1)
        if enc in ENC:
            e = ENC[enc]
            if dec == "unet": return smp.Unet(e, encoder_weights=w, classes=1)
            if dec == "fpn": return smp.FPN(e, encoder_weights=w, classes=1)
            if dec == "seg": return smp.Segformer(e, encoder_weights=w, classes=1)
'''
src = open(f"{W}/train.py").read(); anchor = '    if name == "xattn":'
assert anchor in src and "ENC = {" not in src
open(f"{W}/train.py", "w").write(src.replace(anchor, NEW + anchor, 1))

# --- manifest: train on the clean video-grouped split, evaluate with negatives attached
rows = list(csv.DictReader(open(f"{W}/manifests/v2.csv")))
keep = [r for r in rows if r["split"] in ("train", "val")]
ev = [r for r in csv.DictReader(open(f"{W}/manifests/r2_e4_eval.csv")) if r["split"].startswith(("test_", "neg_"))]
with open(f"{W}/manifests/r5_ablation.csv", "w", newline="") as f:
    wr = csv.DictWriter(f, fieldnames=["image", "mask", "split", "source", "group"], extrasaction="ignore")
    wr.writeheader(); wr.writerows(keep + ev)
import collections; print("r5_ablation.csv", dict(collections.Counter(r["split"] for r in keep + ev)), flush=True)

MISSING = ["unet_mit_b2", "unet_pvtv2b2", "fpn_r34", "fpn_mit_b2", "seg_r34", "seg_pvtv2b2"]   # the 3 other cells are E1 runs
jobs = [dict(job=100 + i, experiment="R5_ablation", manifest="manifests/r5_ablation.csv", model=m, seed=s, epochs=50)
        for i, (s, m) in enumerate((s, m) for s in (42, 43, 44) for m in MISSING)]
with open(f"{W}/jobs.csv", "w", newline="") as f:
    wr = csv.DictWriter(f, fieldnames=list(jobs[0])); wr.writeheader(); wr.writerows(jobs)
print("jobs", len(jobs), jobs[0]["job"], "-", jobs[-1]["job"], flush=True)
import sys
sys.path.insert(0, W)
import torch; from train import build_model          # fail fast if any combination does not build
for m in MISSING:
    n = sum(p.numel() for p in build_model(m, pretrained=False).parameters()) / 1e6
    print(f"  builds: {m:16s} {n:5.1f} M", flush=True)
if os.environ.get("DRY") == "1": sys.exit(0)
SL = {"": ("100-108", "109-117"), "A": ("100-104", "105-108"), "B": ("109-113", "114-117")}[HALF]
cmd = lambda g: (f"CUDA_VISIBLE_DEVICES={g} python {W}/run_queue.py --data-root {ROOT} --jobs {SL[g]} "
                 f"--out /kaggle/working/runs_r5 --budget-hours 11.0 >> /kaggle/working/log_r5_gpu{g}.txt 2>&1")
procs = [subprocess.Popen(cmd(g), shell=True) for g in (0, 1)]; print([p.wait() for p in procs])
print("finished runs", len(glob.glob("/kaggle/working/runs_r5/*/results.json")))
subprocess.run(f"python {W}/aggregate.py /kaggle/working/runs_r5", shell=True)


S1 /kaggle/input/datasets/gokulrocky/chakra-leakbench-v1 
E4 /kaggle/input/datasets/gokulrocky/chakra-leakbench-e4-v1 
CODE /kaggle/input/datasets/gokulrocky/chakra-leakbench-r2-v1-22-09-14-50
r5_ablation.csv {'train': 1228, 'val': 222, 'test_kvasir': 100, 'test_clinicdb': 62, 'test_colondb': 380, 'test_etis': 196, 'test_cvc300': 60, 'neg_polypgen': 4275, 'neg_hyperkvasir': 2947}
jobs 18 100 - 117
  builds: unet_mit_b2       27.5 M
  builds: unet_pvtv2b2      28.1 M
  builds: fpn_r34           23.2 M
  builds: fpn_mit_b2        26.1 M
  builds: seg_r34           21.9 M
  builds: seg_pvtv2b2       25.4 M
[0, 0]
finished runs 18
r5_ablation.csv fpn_mit_b2     n=3 val=0.904±0.004 kvasir=0.920±0.004 clinic=0.904±0.005 colon=0.783±0.004 etis=0.773±0.020 cvc300=0.870±0.002 FA=0.862±0.064
r5_ablation.csv fpn_r34        n=3 val=0.881±0.003 kvasir=0.899±0.009 clinic=0.906±0.004 colon=0.723±0.017 etis=0.687±0.022 cvc300=0.876±0.010 FA=0.322±0.101
r5_ablation.csv seg_pvtv2b2    n=3 val=0.901±0.00

CompletedProcess(args='python /kaggle/working/leakbench/aggregate.py /kaggle/working/runs_r5', returncode=0)